In [57]:
import os
from dotenv import load_dotenv

load_dotenv()  # Reads the .env file
api_key = os.getenv("OPENROUTER_API_KEY")

model = os.getenv("MODEL")
print(f"Loaded model: {model}")  # Debug check: ensure it is not None or empty

Loaded model: openrouter/free


In [58]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openrouter import ChatOpenRouter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool

In [59]:
loader = PyMuPDFLoader(file_path="../data/CorporatePolicyHandbook.pdf")

In [60]:
data = loader.load()


In [61]:
len(data)

7

In [62]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=20)

In [63]:
chunks = splitter.split_documents(data)

In [64]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2421.94it/s]


In [65]:
vector_store = InMemoryVectorStore.from_documents(
    embedding=embeddings,
    documents=chunks
)

In [66]:
llm = ChatOpenRouter(
    model=model,
    api_key=api_key,
    temperature=0.7,
    max_tokens=500,
)

In [67]:
@tool
def get_similar_chunks(query: str) -> str:
    """Search the vector database and retrieve relevant context chunks for a given query.
    
    Args:
        query: The natural language search query.
        
    Returns:
        A combined string of the most relevant document contents.
    """
    results = vector_store.similarity_search(query , k=2)

    similar_data = []
    for doc in results:
        similar_data.append(doc.page_content)
        
    text = " ".join(similar_data)
    return text

In [68]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from uuid import uuid4
SYSTEM_PROMPT = """ 

You are a Corporate Policy Assistant. Your sole job is to answer questions using only the corporate policy handbook.

RULES:
1. Always call `get_similar_chunks(query=...)` to find relevant sections before answering.
2. Answer strictly using facts found in the retrieved excerpts. Never guess or use outside knowledge.
3. If the answer is not in the excerpts, reply: "This information is not covered in the company policy handbook."
4. Keep answers direct, accurate, and concise. Cite specific rules or section names when available.

"""

agent = create_agent(
    model=llm,
    tools=[get_similar_chunks],
    checkpointer=InMemorySaver(),
    system_prompt=SYSTEM_PROMPT
)

In [69]:
thread_id = str(uuid4())
while True:
    query = input("Enter your query")

    if query.lower() in ["exit","quit","bye"]:
        break

    response = agent.invoke(
            {"messages": [{"role": "user", "content": query}]},
            config={"configurable": {"thread_id": thread_id}},
    )

    messages = response.get("messages", [])

    reply = messages[-1].content
    print(f"Agent: {reply}\n")


    

Agent: According to the company policy handbook, the Global Ethics Hotline contact number is:

**1-800-555-ETHIC**

This hotline is toll-free, available 24/7/365, and operated by an independent third-party monitoring agency.

